<a href="https://colab.research.google.com/github/freida20git/bird-detection-tracking/blob/main/tracker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics

In [ ]:
!gdown 'https://drive.google.com/uc?id=1zlntf7zrK9U2y-4ZPd-6LqDyJIfhAj5y'

In [ ]:
!pip install yt-dlp

In [ ]:
# download pretrained models:
!gdown 'https://drive.google.com/uc?id=1x-A9WOyZtZrOqlgM-EXg5g4ek6YgmCYO'

Downloading...
From: https://drive.google.com/uc?id=1x-A9WOyZtZrOqlgM-EXg5g4ek6YgmCYO
To: /content/bestbirdsonly.pt
100% 5.47M/5.47M [00:00<00:00, 28.2MB/s]


In [ ]:
!gdown 'https://drive.google.com/uc?id=16XVu6YHRthELaPRXiG9DRPkbtPOM7r49'

Downloading...
From: https://drive.google.com/uc?id=16XVu6YHRthELaPRXiG9DRPkbtPOM7r49
To: /content/10489468-hd_1080_1920_30fps.mp4
100% 5.40M/5.40M [00:00<00:00, 36.0MB/s]


In [ ]:
!yt-dlp 'https://www.youtube.com/watch?v=e3EDXyoZ-Qw&pp=ygUUZGlzdGFudCBiaXJkcyBmbHlpbmc%3D'

In [ ]:
from google.colab.patches import cv2_imshow
from deep_sort_realtime.deepsort_tracker import DeepSort
import cv2
import json  # Added for JSON support
from ultralytics import YOLO
import numpy as np

In [ ]:
video_path = '/content/10489468-hd_1080_1920_30fps.mp4'

### deepSORT:

In [ ]:
!pip install deep_sort_realtime # Install the deep_sort_realtime library if not already installed

In [ ]:
# Filter predictions based on confidence
from google.colab.patches import cv2_imshow
cap = cv2.VideoCapture(video_path)
model=YOLO('bestbirdsonly.pt')
tracker = DeepSort(max_iou_distance=0.65,
            max_cosine_distance=0.25, # Effectively disable appearance matching
            nms_max_overlap=0.5,
            n_init=2,
            gating_only_position=True,)

# Get video properties for output video
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# Create VideoWriter object
fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Use appropriate codec
out = cv2.VideoWriter('output.mp4', fourcc, fps, (frame_width, frame_height))

frame_number = 0
all_annotations = []  # List to store annotations for all frames

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_number += 1
    results = model.predict(frame, conf=0.3, classes=0, iou=0.4)

    # Create annotations for this frame
    frame_annotations = {
        "frame_number": frame_number,
        "objects": []
    }
    # Extract detections in the format expected by DeepSort
    detections = []
    for *xyxy, conf, cls in results[0].boxes.data:
        x1, y1, x2, y2 = map(int, xyxy)
        detections.append([[x1, y1, x2 - x1, y2 - y1], conf.item(), int(cls.item())])

    tracks = tracker.update_tracks(detections, frame=frame)

    # Draw bounding boxes and labels on the frame
    for track in tracks:
        if not track.is_confirmed():
            continue
        track_id = track.track_id
        ltrb = track.to_ltrb()
        class_id = track.get_det_class()
        x1, y1, x2, y2 = map(int, ltrb)
        confidence = track.det_conf
        if confidence is not None:
            frame_annotations["objects"].append({
                "track_id": track_id,
                "class_id": class_id,
                "class_name": model.names[class_id],  # Get class name from model
                "confidence": confidence,
                "bbox": {
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2
                }
            })
            text = f"{track_id} - {model.names[class_id]}"
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, text, (x1, y1 - 10), cv2.FONT_HERSHEY_DUPLEX, 0.9, (0, 255, 0), 2)

    all_annotations.append(frame_annotations)  # Add frame annotations to the list

    # Write the frame to the output video
    out.write(frame)

# Save annotations to JSON file after processing all frames
with open('desert_pred_ds.json', 'w') as f:
    json.dump(all_annotations, f, indent=2)
print("Annotations saved to sunset_pred_ds.json")

# Release resources
cap.release()
out.release()
cv2.destroyAllWindows()

### SORT: cloned from https://raw.githubusercontent.com/abewley/sort/master/sort.py with minor modifications

In [ ]:
!pip install ultralytics filterpy opencv-python

In [ ]:
!wget https://raw.githubusercontent.com/freida20git/bird-detection-tracking/main/sort.py

In [ ]:
import cv2
import numpy as np
import json
from ultralytics import YOLO
from sort import *

# Load video and model
cap = cv2.VideoCapture(video_path)
model = YOLO('bestbirdsonly.pt')

# Initialize SORT tracker
KalmanBoxTracker.count = 0
tracker = Sort(max_age=30, min_hits=3, iou_threshold=0.1)

# Video writer setup
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
out = cv2.VideoWriter('output_sort.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))

# Annotation storage
frame_number = 0
all_annotations = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_number += 1
    results = model(frame, conf=0.3, classes=0)

    detections = []
    det_info = []
    for box in results[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])
        cls_id = int(box.cls[0])
        detections.append([x1, y1, x2, y2, conf])
        det_info.append({"bbox": (x1, y1, x2, y2), "conf": conf, "class_id": cls_id})

    dets_np = np.array(detections) if detections else np.empty((0, 5))
    track_bbs_ids = tracker.update(dets_np)

    frame_annotations = {"frame_number": frame_number, "objects": []}
    if len(track_bbs_ids) > 3:
      print("more than 3 birds detected in frame: ", frame_number)
    for track in track_bbs_ids:
        x1, y1, x2, y2, track_id = track.astype(int)

        # Match SORT box to closest YOLO detection (within 20px)
        matched_det = None
        for det in det_info:
            dx1, dy1, dx2, dy2 = det["bbox"]
            if abs(dx1 - x1) < 20 and abs(dy1 - y1) < 20:
                matched_det = det
                break

        if matched_det:
            class_id = matched_det["class_id"]
            confidence = matched_det["conf"]
            class_name = model.names[class_id]
        else:
            class_id = 0
            confidence = 0.0
            class_name = "Bird"

        frame_annotations["objects"].append({
            "track_id": int(track_id),
            "class_id": int(class_id),
            "class_name": str(class_name),
            "confidence": float(confidence),
            "bbox": {"x1": int(x1), "y1": int(y1), "x2": int(x2), "y2": int(y2)}
        })

        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.putText(frame, f"ID {track_id} {class_name}", (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

    all_annotations.append(frame_annotations)
    out.write(frame)

# Save annotations to JSON
with open('sort_desert_annotations.json', 'w') as f:
    json.dump(all_annotations, f, indent=2)

# Release resources
cap.release()
out.release()
cv2.destroyAllWindows()
print("Tracking complete. Output video and annotations saved.")


### bytetrack/botsort:

In [ ]:
def byte_or_bot(tracker, output_vid, json_file):
    # Load video and model
    cap = cv2.VideoCapture(video_path)
    model = YOLO('bestbirdsonly.pt')

    # Get video properties for output video
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    # Create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Use appropriate codec
    out = cv2.VideoWriter(output_vid, fourcc, fps, (frame_width, frame_height))

    frame_number = 0
    all_annotations = []  # List to store annotations for all frames

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_number += 1
        # Run YOLO detection with tracker
        results = model.track(frame, conf=0.3, tracker=tracker, persist=True)

        # Create annotations for this frame
        frame_annotations = {
            "frame_number": frame_number,
            "objects": []
        }

        # Check if there are any detections
        if results[0].boxes.id is not None:
            # Get tracking IDs
            track_ids = results[0].boxes.id.int().tolist()

            # Get bounding boxes, confidences, and class IDs
            boxes = results[0].boxes.xyxy.int().tolist()
            confidences = results[0].boxes.conf.tolist()
            class_ids = results[0].boxes.cls.int().tolist()

            # Process each detection
            for box, track_id, confidence, class_id in zip(boxes, track_ids, confidences, class_ids):
                x1, y1, x2, y2 = box

                # Add to annotations
                frame_annotations["objects"].append({
                    "track_id": int(track_id),
                    "class_id": int(class_id),
                    "class_name": model.names[int(class_id)],  # Get class name from model
                    "confidence": float(confidence),
                    "bbox": {
                        "x1": int(x1),
                        "y1": int(y1),
                        "x2": int(x2),
                        "y2": int(y2)
                    }
                })

                # Draw bounding box and label
                text = f"{track_id} - {model.names[int(class_id)]} ({confidence:.2f})"
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, text, (x1, y1 - 10), cv2.FONT_HERSHEY_DUPLEX, 0.9, (0, 255, 0), 2)

        all_annotations.append(frame_annotations)  # Add frame annotations to the list

        # Write the frame to the output video
        out.write(frame)

        # Display the frame (optional)
        # cv2_imshow(frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Save annotations to JSON file after processing all frames
    with open(json_file, 'w') as f:
        json.dump(all_annotations, f, indent=2)
    print("Annotations saved to sunset_annotations_byte.json")

    # Release resources
    cap.release()
    out.release()
    cv2.destroyAllWindows()
byte_or_bot('bytetrack.yaml', 'output_byte.mp4', 'desert_annotations_byte.json')
byte_or_bot('botsort.yaml', 'output_bot.mp4', 'desert_annotations_bot.json')

# video output:

In [ ]:
!rm "/content/result_compressed.mp4"

In [ ]:
from IPython.display import HTML
from base64 import b64encode
import os

# Input video path
save_path = '/content/output.mp4'

compressed_path = "/content/result_compressed.mp4"

os.system(f"ffmpeg -i {save_path} -vcodec libx264 {compressed_path}")

# Show video
mp4 = open(compressed_path,'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=400 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)